In [1]:
import time
import torch
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

c:\Users\mohamed\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Anlamsal benzerlik modelini başlatıyoruz
print("Anlamsal benzerlik modeli yükleniyor...")
anlamsal_benzerlik_modeli = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

# 2. Akıllı Cevap Kontrolü Fonksiyonu
def cevap_dogru_mu(dogru_cevap_index, verilen_cevap, secenekler):
    harfler = ['A', 'B', 'C', 'D', 'E']
    dogru_harf = harfler[dogru_cevap_index]
    verilen_cevap = verilen_cevap.upper().strip()

    if dogru_harf == verilen_cevap:
        return True
    elif len(verilen_cevap) > 1 and verilen_cevap[1] in [" ", ":", ")", "=", "-", "."]:
        return dogru_harf == verilen_cevap[0]
    else:
        encoded_cevap = anlamsal_benzerlik_modeli.encode([verilen_cevap])
        encoded_secenekler = anlamsal_benzerlik_modeli.encode(secenekler)
        benzerlik_listesi = anlamsal_benzerlik_modeli.similarity(encoded_cevap, encoded_secenekler).tolist()[0]
        en_yuksek_benzerlik_index = benzerlik_listesi.index(max(benzerlik_listesi))
        return en_yuksek_benzerlik_index == dogru_cevap_index

# 3. MMLU Veri Setini Yükleme
print("MMLU Veri seti indiriliyor...")
mmlu_veri = pd.read_parquet("hf://datasets/alibayram/yapay_zeka_turkce_mmlu_model_cevaplari/data/train-00000-of-00001.parquet")

# 4. Hugging Face Modeli İle Test Koşturma Fonksiyonu
def hf_modeli_test_et(model_id, test_limiti=None):
    print(f"\n--- {model_id} Modeli Yükleniyor ---")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    baslama_zamani = time.time()
    dogru_cevap_sayisi = 0
    toplam_soru = test_limiti if test_limiti else len(mmlu_veri)

    for i in range(toplam_soru):
        soru_metni = mmlu_veri.iloc[i]['soru'] + "\n"
        harfler = ['A', 'B', 'C', 'D', 'E']
        for j, secenek in enumerate(mmlu_veri.iloc[i]['secenekler']):
            soru_metni += f"{harfler[j]}: {secenek}\n"

        prompt = f"Sana soru ve seçenekleri veriyorum. sadece hangi seçeneğin sorunun doğru cevabı olduğunu yaz. Örneğin 'A' veya 'B' gibi. Lütfen herhangi bir açıklama yapma!\nSoru: {soru_metni}\nCevap:"

        # Tokenizer & Model Çıkarımı (Inference)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=10, 
                temperature=0.01, # Deterministik yanıt için
                do_sample=False
            )
        
        # Sadece modelin yeni ürettiği cevabı alıyoruz
        gelen_metin = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        sonuc = cevap_dogru_mu(mmlu_veri.iloc[i]['cevap'], gelen_metin, mmlu_veri.iloc[i]['secenekler'])
        if sonuc:
            dogru_cevap_sayisi += 1

        simdi = time.time()
        gecen_sure = round(simdi - baslama_zamani, 2)
        basari_orani = round((dogru_cevap_sayisi / (i + 1)) * 100, 2)
        print(f"\rSoru: {i+1}/{toplam_soru} | Doğru: {dogru_cevap_sayisi} | Başarı: %{basari_orani} | Süre: {gecen_sure}s", end="")

    toplam_sure = round(time.time() - baslama_zamani, 2)
    genel_basari = round((dogru_cevap_sayisi / toplam_soru) * 100, 2)
    
    print(f"\n\n✅ {model_id} Testi Tamamlandı!")
    print(f"Başarı Oranı: %{genel_basari} | Toplam Süre: {toplam_sure} saniye")
    
    return {"model": model_id, "basari": genel_basari, "sure": toplam_sure}

Anlamsal benzerlik modeli yükleniyor...


c:\Users\mohamed\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mohamed\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<0

MMLU Veri seti indiriliyor...


In [3]:
benim_sonucum = hf_modeli_test_et("Endezyar/gemma-3-finetune")


--- Endezyar/gemma-3-finetune Modeli Yükleniyor ---


c:\Users\mohamed\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mohamed\.cache\huggingface\hub\models--Endezyar--gemma-3-finetune. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 10

AssertionError: Torch not compiled with CUDA enabled